<a href="https://colab.research.google.com/github/Adhiaris/Grokking-Deep-Learning/blob/main/chapter_10_convolutional_neural_networks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 10: Neural Networks That Learn About Edges and Corners
### Introduction to Convolutional Neural Networks

Convolutional layers detect local patterns (edges, corners, textures) by reusing the same small weight matrix (called a **kernel** or **filter**) across all positions in the input. This weight sharing dramatically reduces parameters and makes the network translation-invariant.

## 1. The Key Idea: Reusing Weights

In a fully-connected layer, every input connects to every output with a unique weight. In a convolutional layer, one small filter slides across the entire input — the same weights are shared at every position.

In [ ]:
import numpy as np

input_size  = 8
filter_size = 3

fc_params   = input_size * input_size
conv_params = filter_size * filter_size

print("Parameter comparison (1D simplified):")
print(f"  Fully-connected layer : {fc_params} weights")
print(f"  Convolutional layer   : {conv_params} weights (shared across all positions)")
print(f"  Reduction factor      : {fc_params // conv_params}x")

## 2. 1D Convolution

A 1D convolution slides a filter of length k over a 1D input. At each position, it computes the dot product of the filter with the current window of the input.

In [ ]:
import numpy as np

def convolve1d(input_seq, kernel):
    k   = len(kernel)
    out = []
    for i in range(len(input_seq) - k + 1):
        window = input_seq[i:i+k]
        out.append(np.dot(window, kernel))
    return np.array(out)

signal = np.array([0, 0, 0, 1, 1, 1, 0, 0, 0], dtype=float)
edge_kernel = np.array([-1, 1, 0], dtype=float)

result = convolve1d(signal, edge_kernel)

print("Input signal:", signal)
print("Edge kernel :", edge_kernel)
print("Convolution :", result)
print("\nPositive values mark rising edges, negative mark falling edges.")

## 3. 2D Convolution

For images, the filter is a small 2D matrix (e.g., 3×3) that slides over the 2D pixel grid.

In [ ]:
import numpy as np

def convolve2d(image, kernel):
    kh, kw = kernel.shape
    ih, iw = image.shape
    out    = np.zeros((ih - kh + 1, iw - kw + 1))
    for i in range(out.shape[0]):
        for j in range(out.shape[1]):
            out[i, j] = np.sum(image[i:i+kh, j:j+kw] * kernel)
    return out

image = np.array([
    [0,0,0,0,0],
    [0,1,1,1,0],
    [0,1,1,1,0],
    [0,1,1,1,0],
    [0,0,0,0,0],
], dtype=float)

sharpen = np.array([
    [ 0, -1,  0],
    [-1,  5, -1],
    [ 0, -1,  0],
], dtype=float)

output = convolve2d(image, sharpen)

print("Input image (5x5):")
print(image.astype(int))
print("\nConvolution output (3x3):")
print(output)

## 4. Multiple Filters

A convolutional layer uses multiple filters simultaneously. Each filter learns to detect a different feature (horizontal edge, vertical edge, color blob, etc.).

In [ ]:
import numpy as np

np.random.seed(0)

image       = np.random.rand(8, 8)
num_filters = 4
filter_size = 3

filters = np.random.rand(num_filters, filter_size, filter_size) - 0.5

def convolve2d(image, kernel):
    kh, kw = kernel.shape
    ih, iw = image.shape
    out = np.zeros((ih - kh + 1, iw - kw + 1))
    for i in range(out.shape[0]):
        for j in range(out.shape[1]):
            out[i, j] = np.sum(image[i:i+kh, j:j+kw] * kernel)
    return out

feature_maps = []
for f in range(num_filters):
    feature_map = convolve2d(image, filters[f])
    feature_maps.append(feature_map)
    print(f"Filter {f} -> feature map shape: {feature_map.shape}")

print(f"\nTotal feature maps: {len(feature_maps)} (one per filter)")

## 5. Why CNNs Reduce Overfitting

Compared to a fully-connected layer on the same input, a convolutional layer has far fewer parameters. Fewer parameters means the network has less capacity to memorize noise.

In [ ]:
image_h, image_w = 28, 28
num_output       = 10
filter_size      = 3
num_filters      = 16

fc_params   = image_h * image_w * num_output
conv_params = filter_size * filter_size * num_filters

print(f"Task: {image_h}x{image_w} image -> {num_output} class output")
print(f"\nFully-connected parameters : {fc_params:,}")
print(f"Convolutional parameters   : {conv_params:,}")
print(f"Reduction                  : {fc_params / conv_params:.0f}x fewer parameters")